# Model Training Pipeline

## Data Preparation

### PDF Content Extraction

We have a PDF autobiography of Jesse Livermore called *Reminiscences of a Stock Operator* here.

Given that the PDF has chapters labeled already, let's keep only chapters and remove unnecessary parts.

In [1]:
from PyPDF2 import PdfReader

# Load the PDF file
pdf_path = "Reminiscences of a Stock Operator 2008.pdf"
reader = PdfReader(pdf_path)

outlines = reader.outline

In [2]:
def print_chapter_titles(outlines):
    for item in outlines:
        if isinstance(item, list):
            print_chapter_titles(item)  # handle nested outlines
        else:
            print(item.title)

print_chapter_titles(outlines)

Front
Title
Copyright
Dedication
Index
I
II
III
IV
V
VI
VII
VIII
IX
X
XI
XII
XIII
XIV
XV
XVI
XVII
XVIII
XIX
XX
XXI
XXII
XXIII
XXIV
Back


In [3]:
chapter_info = []
for item in outlines:
    title = item.title
    page_number = reader.get_destination_page_number(item)
    chapter_info.append((title, page_number))
print(chapter_info)

[('Front\r', 0), ('Title', 2), ('Copyright', 3), ('Dedication', 4), ('Index', 6), ('I', 8), ('II', 22), ('III', 38), ('IV', 48), ('V', 66), ('VI', 80), ('VII', 92), ('VIII', 100), ('IX', 114), ('X', 132), ('XI', 146), ('XII', 160), ('XIII', 178), ('XIV', 192), ('XV', 210), ('XVI', 220), ('XVII', 238), ('XVIII', 254), ('XIX', 264), ('XX', 278), ('XXI', 286), ('XXII', 304), ('XXIII', 326), ('XXIV', 338), ('Back\r', 347)]


In [4]:
chapter_info = chapter_info[5:-1]

In [5]:
chapter_info

[('I', 8),
 ('II', 22),
 ('III', 38),
 ('IV', 48),
 ('V', 66),
 ('VI', 80),
 ('VII', 92),
 ('VIII', 100),
 ('IX', 114),
 ('X', 132),
 ('XI', 146),
 ('XII', 160),
 ('XIII', 178),
 ('XIV', 192),
 ('XV', 210),
 ('XVI', 220),
 ('XVII', 238),
 ('XVIII', 254),
 ('XIX', 264),
 ('XX', 278),
 ('XXI', 286),
 ('XXII', 304),
 ('XXIII', 326),
 ('XXIV', 338)]

Let's store the chapters.

In [6]:
chapters = []

for idx, (title, start_page) in enumerate(chapter_info):
    end_page = chapter_info[idx + 1][1] if idx + 1 < len(chapter_info) else len(reader.pages)
    
    chapter_text = ""
    for p in range(start_page, end_page):
        chapter_text += reader.pages[p].extract_text() + "\n"
    chapters += [chapter_text]
    
    print(f"=== {title} ===")
    print(chapter_text[:500])  # Print the first 500 characters of chapter
    print("\n\n")

=== I ===
I
I went to work when I was just out of grammar school. I  
got  a  job  as  quotation-board  boy  in  a  stock-brokerage  
office. I was quick at figures. At school I did three years of  
arithmetic  in  one.  I  was  particularly  good  at  mental  
arithmetic. As quotation-board boy I posted the numbers  
on  the  big  board  in  the  customers'  room.  One  of  the  
customers  usually  sat  by  the  ticker  and  called  out  the  
prices. They couldn't come too fast for me. I have always  




=== II ===
II
Between  the  discovery  that  the  Cosmopolitan  Stock  
Brokerage Company was ready to beat me by foul means if  
the killing handicap of a three-point margin and a point-
and-a-half premium didn't do it, and hints that they didn't  
want my business anyhow, I soon  made up my mind to go  
to New York, where I could trade in the office of some  
member of the New York Stock Exchange. I didn't want  
any  Boston  branch,  where  the  quotations  had  to  be  
telegra

In [7]:
for i in range(1, len(chapters) + 1):
    print("Chapter ", i, ": ", end="", sep="")
    print(len(chapters[i - 1]))

Chapter 1: 26540
Chapter 2: 29821
Chapter 3: 18192
Chapter 4: 32266
Chapter 5: 25142
Chapter 6: 19794
Chapter 7: 13398
Chapter 8: 27079
Chapter 9: 34163
Chapter 10: 28749
Chapter 11: 24862
Chapter 12: 30788
Chapter 13: 26403
Chapter 14: 35009
Chapter 15: 18089
Chapter 16: 31694
Chapter 17: 32005
Chapter 18: 17238
Chapter 19: 25356
Chapter 20: 14582
Chapter 21: 33713
Chapter 22: 41289
Chapter 23: 22556
Chapter 24: 9868


Let's split up the content in each chapter, and feed them respectively into the model.

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

chunks = []
for chapter in chapters:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=300
    )
    chunks += text_splitter.split_text(chapter)
chunks[0]

"I\nI went to work when I was just out of grammar school. I  \ngot  a  job  as  quotation-board  boy  in  a  stock-brokerage  \noffice. I was quick at figures. At school I did three years of  \narithmetic  in  one.  I  was  particularly  good  at  mental  \narithmetic. As quotation-board boy I posted the numbers  \non  the  big  board  in  the  customers'  room.  One  of  the  \ncustomers  usually  sat  by  the  ticker  and  called  out  the  \nprices. They couldn't come too fast for me. I have always  \nremembered figures. No trouble at all.\nThere were plenty of other employees in that office. Of"

### Prepare to Generate Q&A Pairs

Now that we have all the chapters, let's prepare generate Q&A pairs.

We will use ollama with a local model, but we can switch to something else later on if needed.

Let's create a function called ChatWithOllama. We will build more functions based on this.

In [9]:
from ollama import chat
from ollama import ChatResponse

def ChatWithOllama(systemContent="You are a kindergarten teacher who always use easy words to answer questions.", userContent='Why is the sky blue?', model='qwen3:8b', showThink=True):
    response: ChatResponse = chat(
        model=model, 
        messages=[
            {
                'role': 'system',
                'content': systemContent,
                
            }, 
             {
                'role': 'user',
                'content': userContent,
            }
        ]
    )

    # print(response.message.content)

    result = response.message.content

    if not showThink:
        import re
        result = re.sub(r"<think>.*?</think>\s*", "", result, flags=re.DOTALL)

    return result

def ChatWith(framework="Ollama", systemContent="You are a kindergarten teacher who always use easy words to answer questions.", userContent='Why is the sky blue?', model='qwen3', showThink=True):
    if framework == "Ollama":
        return ChatWithOllama(systemContent=systemContent, userContent=userContent, model=model, showThink=showThink)

For Json based result specifically:

In [10]:
from ollama import chat
from pydantic import BaseModel

class Country(BaseModel):
    name: str
    capital: str
    languages: list[str]

class CountryList(BaseModel):
    countries: list[Country] 


def jsonWithOllama(jsonClass=CountryList, systemContent="You are a kindergarten teacher who always use easy words to answer questions.", userContent='Tell me about Canada and Mexico. Respond as a JSON array of countries.', model='qwen3:8b'):
    response = chat(
        # messages=[
        #     {'role': 'user', 'content': 'Tell me about Canada and France. Respond as a JSON array of countries.'}
        # ],
        messages=[
            {
                'role': 'system',
                'content': systemContent,
                
            }, 
             {
                'role': 'user',
                'content': userContent,
            }
        ],
        model=model,
        # format=Country.model_json_schema(),  # ✅ 正确用法
        format = jsonClass.model_json_schema(),
    )
    
    # print(response.message.content)
    # country = Country.model_validate_json(response.message.content)
    # country = TypeAdapter(CountryList).validate_json(response.message.content)
    try:
        answers = jsonClass.model_validate_json(response.message.content)
    except Exception as e:
        print("Error parsing JSON response:", e)
        print("Prompt:", userContent)
        print("Raw response:", response.message.content)
        answers = None
    # print(answers)

    return answers

jsonWithOllama()

CountryList(countries=[Country(name='Canada', capital='Ottawa', languages=['English', 'French']), Country(name='Mexico', capital='Mexico City', languages=['Spanish'])])

And some functions for different purposes

In [11]:
import json

def generalizeWithLLM(text):
    systemContent="""
        You are a thoughtful assistant skilled at abstracting long, detailed texts into their most important general ideas.
    
        Please read the following text and write a generalization that:
        - Extracts the core themes or ideas,
        - Avoids specific numbers, names, or examples unless essential,
        - Focuses on the overall message or logic of the content,
        - Uses simple and clear language, suitable for someone unfamiliar with the subject.
    """

    userContent = f"""
        Please read the following text and write a generalization that:
        - Extracts the core themes or ideas
        - Avoids specific names, numbers, or examples unless essential
        - Focuses on the overall message or logic
        - Uses simple, clear language suitable for someone new to the topic
        
        Text:
        {text}
        

        Return the generalization as bullet points if the ideas are distinct.
    """

    showThink = False

    
    return ChatWith(systemContent=systemContent, userContent=userContent, showThink=showThink)

def generateAQuestionBasedOnAnswer(answer):
    systemContent = "You are an assistant that creates a single, well-phrased question based on multiple bullet-point answers. The question should be clear and reflect the overall idea covered by the bullet points."

    userContent = f"""
    Here is a list of bullet points:

    {answer}

    Write one question that would lead to this answer.
    """

    showThink = False

    return ChatWith(systemContent=systemContent, userContent=userContent, showThink=showThink)


from pydantic import BaseModel
class Q_and_A_pair(BaseModel):
    question: str
    answer: str
class Q_and_A_pair_List(BaseModel):
    Q_and_A_pairs: list[Q_and_A_pair] 
    
def generateQuestionAndAnswerBasedOnChunk(chunk, jsonClass=Q_and_A_pair_List):
    systemContent = """
        You are an intelligent assistant trained to extract insightful question and answer pairs from autobiographical texts, especially those related to finance and personal experiences in the stock market. Your job is to extract as many useful and specific question-answer pairs as possible from the provided text. The questions should focus on key events, decisions, lessons learned, emotional states, strategies, and historical context described in the text.
        Only respond with a well-formatted JSON array. Each element should be an object with a "question" and an "answer" field.
    """

    userContent = f"""
    Please extract question and answer pairs from the following text:

    {chunk}
    """

    result = jsonWithOllama(jsonClass=jsonClass, systemContent=systemContent, userContent=userContent)
    
    if result is None:
        return []
    else:
        return result.Q_and_A_pairs
    
    

In [12]:
chunks[20]

'"How do you mean, play it?" I asked. To me the only  \npeople who played or could play tips were the customers  \nold jiggers with oodles of dough. Why, it cost hundreds,  \neven thousands of dollars, to get into the game. It was like  \nowning your private carriage and having a coachman who  \nwore a silk hat.\n"That\'s what I mean; play it!" he said.\n"How much you got?"\n"How much you need?"\n"Well, I can trade in five shares by putting up $5."\n"How are you going to play it?"\n"I\'m going to buy all the Burlington the bucket shop will  \nlet me carry with the money I give him for margin," he'

In [13]:
a = generateQuestionAndAnswerBasedOnChunk(chunks[20])

In [14]:
len(a)

5

In [15]:
a[0]

Q_and_A_pair(question="What was the speaker's initial understanding of 'play it'?", answer="The speaker initially thought that 'play it' referred to playing tips, which required significant financial investment, such as hundreds or even thousands of dollars.")

In [16]:
a[0].question

"What was the speaker's initial understanding of 'play it'?"

In [17]:
a[0].answer

"The speaker initially thought that 'play it' referred to playing tips, which required significant financial investment, such as hundreds or even thousands of dollars."

### Q&A Generation

Now that we have everything prepared, let's do the Q&A generation

If QA_Pair.csv already exists locally, we will not do the generation

In [18]:
import os
import pandas as pd
import time
import csv

filename = "QA_Pair.csv"
if not os.path.exists(filename):
    qa_pairs = []

    def appendQA(question, answer):
        global qa_pairs
        qa_pairs += [{
            "question": question, 
            "answer": answer
        }]

    # First let's do the generalization for each chapter
    chapterCounter = 1
    totalChapterNumber = 24
    for chapter in chapters:
        startTime = time.time()
        answer = generalizeWithLLM(chapter)
        question = generateAQuestionBasedOnAnswer(answer)

        appendQA(question, answer)
        endTime = time.time()
        
        print("Chapter: (", chapterCounter, "/", totalChapterNumber, ") Done!", " Took: ", endTime-startTime, sep="")
        chapterCounter += 1

    # Now let's do chunks
    for chunkCounter in range(0, len(chunks)):
        startTime = time.time()

        q_and_a_pairs = generateQuestionAndAnswerBasedOnChunk(chunks[chunkCounter])

        for q_and_a_pair in q_and_a_pairs:
            question = q_and_a_pair.question
            answer = q_and_a_pair.answer
            appendQA(question, answer)
        
        endTime = time.time()
        
        print("Chunk: (", chunkCounter + 1, "/", len(chunks), ") Done!", " Took: ", endTime-startTime, sep="")
        # print(qa_pairs)

    
    # Save everything
    print(qa_pairs)
    df = pd.DataFrame(qa_pairs)
    df.to_csv(filename, index=False)
    
    
else:
    print(f"{filename} already exists. Skipping file creation.")


Chapter: (1/24) Done! Took: 41.61542820930481
Chapter: (2/24) Done! Took: 29.75993847846985
Chapter: (3/24) Done! Took: 29.51904582977295
Chapter: (4/24) Done! Took: 33.126193046569824
Chapter: (5/24) Done! Took: 29.4214506149292
Chapter: (6/24) Done! Took: 38.54589557647705
Chapter: (7/24) Done! Took: 35.196335792541504
Chapter: (8/24) Done! Took: 34.94249653816223
Chapter: (9/24) Done! Took: 42.80623126029968
Chapter: (10/24) Done! Took: 35.482885122299194
Chapter: (11/24) Done! Took: 33.22049140930176
Chapter: (12/24) Done! Took: 48.702556133270264
Chapter: (13/24) Done! Took: 40.66453409194946
Chapter: (14/24) Done! Took: 31.731744050979614
Chapter: (15/24) Done! Took: 37.35134840011597
Chapter: (16/24) Done! Took: 32.672977447509766
Chapter: (17/24) Done! Took: 32.2650785446167
Chapter: (18/24) Done! Took: 33.321972608566284
Chapter: (19/24) Done! Took: 32.00351309776306
Chapter: (20/24) Done! Took: 36.795470237731934
Chapter: (21/24) Done! Took: 47.17324471473694
Chapter: (22/24)